# YOLO12n Sidewalk Hazard Training, Minimal Colab Notebook

This is the minimum setup needed to start training YOLO12n on your already unzipped COCO style dataset.

Expected dataset structure:

```text
sidewalk_hazard_dataset/
  annotations/
    train.json
    val.json
  images/
    train/
    val/
  class_map.json
```

This notebook converts the COCO JSON bounding boxes into YOLO `.txt` labels, creates a `data.yaml`, then starts training `yolo12n.pt`.

## 1. Mount Google Drive

Use this if your dataset is in Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Install Ultralytics

In [ ]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.2 MB/s eta 0:00:00


## 3. Set your dataset path

Change `dataset_root` only if your folder is somewhere else.

In [ ]:
from pathlib import Path

# Your already unzipped dataset folder
dataset_root = Path("/content/drive/MyDrive/sidewalk_hazard_dataset")

train_json = dataset_root / "annotations" / "train.json"
val_json = dataset_root / "annotations" / "val.json"
train_images = dataset_root / "images" / "train"
val_images = dataset_root / "images" / "val"

print("Dataset root:", dataset_root)
print("Train JSON exists:", train_json.exists())
print("Val JSON exists:", val_json.exists())
print("Train image folder exists:", train_images.exists())
print("Val image folder exists:", val_images.exists())

assert dataset_root.exists(), "dataset_root does not exist. Fix the path."
assert train_json.exists(), "Missing annotations/train.json"
assert val_json.exists(), "Missing annotations/val.json"
assert train_images.exists(), "Missing images/train folder"
assert val_images.exists(), "Missing images/val folder"

Dataset root: /content/drive/MyDrive/sidewalk_hazard_dataset
Train JSON exists: True
Val JSON exists: True
Train image folder exists: True
Val image folder exists: True


## 4. Read classes from the annotation files

This uses the `categories` section inside `train.json` and creates YOLO class IDs starting at 0.

In [ ]:
import json

with open(train_json, "r") as f:
    train_data = json.load(f)

categories = sorted(train_data["categories"], key=lambda x: x["id"])
category_id_to_yolo_id = {}
names = {}

for yolo_id, category in enumerate(categories):
    category_id_to_yolo_id[category["id"]] = yolo_id
    names[yolo_id] = category.get("name", f"class{yolo_id}")

print("Classes:")
for class_id, class_name in names.items():
    print(class_id, class_name)

Classes:
0 person
1 bicycle
2 car
3 traffic_cone
4 pothole
5 open_manhole
6 pole
7 sign
8 trash_can
9 roadblock
10 tree
11 animal
12 fire_hydrant
13 crosswalk
14 tactile_paving
15 surface_damage


## 5. Convert COCO JSON labels to YOLO labels

YOLO needs one `.txt` label file per image. This cell creates:

```text
labels/train/*.txt
labels/val/*.txt
```

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def index_images(image_dir):
    image_lookup = {}

    for p in Path(image_dir).rglob("*"):
        if p.suffix.lower() in image_extensions:
            image_lookup[p.name] = p
            image_lookup[p.stem] = p

    return image_lookup


def convert_coco_json_to_yolo_fast(json_path, image_dir, label_dir, category_id_to_yolo_id):
    label_dir.mkdir(parents=True, exist_ok=True)

    print("Loading JSON:", json_path)
    with open(json_path, "r") as f:
        data = json.load(f)

    print("Indexing images:", image_dir)
    image_lookup = index_images(image_dir)
    print("Images found in folder:", len(image_lookup) // 2)

    images = {}
    for img in data["images"]:
        images[img["id"]] = img

    labels_by_image = defaultdict(list)
    skipped_annotations = 0

    print("Converting annotations...")
    for ann in data["annotations"]:
        image_id = ann.get("image_id")
        category_id = ann.get("category_id")
        bbox = ann.get("bbox")

        if image_id not in images or category_id not in category_id_to_yolo_id:
            skipped_annotations += 1
            continue

        if bbox is None or len(bbox) != 4:
            skipped_annotations += 1
            continue

        x, y, width, height = bbox

        if x is None or y is None or width is None or height is None:
            skipped_annotations += 1
            continue

        if width <= 0 or height <= 0:
            skipped_annotations += 1
            continue

        img = images[image_id]
        img_width = img.get("width")
        img_height = img.get("height")

        if img_width is None or img_height is None or img_width <= 0 or img_height <= 0:
            skipped_annotations += 1
            continue

        x_center = (x + width / 2) / img_width
        y_center = (y + height / 2) / img_height
        norm_width = width / img_width
        norm_height = height / img_height

        x_center = max(0, min(1, x_center))
        y_center = max(0, min(1, y_center))
        norm_width = max(0, min(1, norm_width))
        norm_height = max(0, min(1, norm_height))

        yolo_class_id = category_id_to_yolo_id[category_id]
        line = f"{yolo_class_id} {x_center:.6f} {y_center:.6f} {norm_width:.6f} {norm_height:.6f}"
        labels_by_image[image_id].append(line)

    missing_images = 0
    labels_written = 0

    print("Writing labels...")
    for image_id, img in images.items():
        file_name = Path(img["file_name"]).name
        stem = Path(file_name).stem

        image_path = image_lookup.get(file_name)
        if image_path is None:
            image_path = image_lookup.get(stem)

        if image_path is None:
            missing_images += 1
            continue

        label_path = label_dir / f"{image_path.stem}.txt"
        lines = labels_by_image.get(image_id, [])

        with open(label_path, "w") as f:
            if len(lines) > 0:
                f.write("\n".join(lines) + "\n")

        labels_written += 1

    print("Converted:", json_path)
    print("Images in JSON:", len(images))
    print("Images with label files written:", labels_written)
    print("Images missing from folder:", missing_images)
    print("Skipped bad annotations:", skipped_annotations)
    print("Labels saved to:", label_dir)


convert_coco_json_to_yolo_fast(
    train_json,
    train_images,
    dataset_root / "labels" / "train",
    category_id_to_yolo_id
)

convert_coco_json_to_yolo_fast(
    val_json,
    val_images,
    dataset_root / "labels" / "val",
    category_id_to_yolo_id
)

Loading JSON: /content/drive/MyDrive/sidewalk_hazard_dataset/annotations/train.json


KeyboardInterrupt: 

## 6. Create YOLO `data.yaml`

In [ ]:
import yaml

data_yaml = {
    "path": str(dataset_root),
    "train": "images/train",
    "val": "images/val",
    "names": names,
}

data_yaml_path = dataset_root / "data.yaml"

with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(data_yaml_path)
print(data_yaml)

/content/drive/MyDrive/sidewalk_hazard_dataset/data.yaml
{'path': '/content/drive/MyDrive/sidewalk_hazard_dataset', 'train': 'images/train', 'val': 'images/val', 'names': {0: 'person', 1: 'bicycle', 2: 'car', 3: 'traffic_cone', 4: 'pothole', 5: 'open_manhole', 6: 'pole', 7: 'sign', 8: 'trash_can', 9: 'roadblock', 10: 'tree', 11: 'animal', 12: 'fire_hydrant', 13: 'crosswalk', 14: 'tactile_paving', 15: 'surface_damage'}}


## 7. Quick sanity check

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


train_image_count = len([p for p in train_images.iterdir() if p.suffix.lower() in image_extensions])
val_image_count = len([p for p in val_images.iterdir() if p.suffix.lower() in image_extensions])
train_label_count = len(list((dataset_root / "labels" / "train").glob("*.txt")))
val_label_count = len(list((dataset_root / "labels" / "val").glob("*.txt")))

print("Train images:", train_image_count)
print("Train labels:", train_label_count)
print("Val images:", val_image_count)
print("Val labels:", val_label_count)

assert train_image_count > 0, "No training images found."
assert val_image_count > 0, "No validation images found."
assert train_label_count > 0, "No training labels were created."
assert val_label_count > 0, "No validation labels were created."

Train images: 12102
Train labels: 11636
Val images: 2909
Val labels: 2909


## 8. Train YOLO12n

This starts training. Use `device=0` if Colab gives you a GPU. Use `device="cpu"` only if you have no GPU.

In [ ]:
from ultralytics import YOLO
import torch

training_device = 0 if torch.cuda.is_available() else "cpu"
print("Training device:", training_device)

model = YOLO("yolo12n.pt")

results = model.train(
    data=str(data_yaml_path),
    epochs=3,
    imgsz=640,
    batch=16,
    device=training_device,
    project="/content/drive/MyDrive/yolo12n_sidewalk_runs",
    name="yolo12n_sidewalk_hazard",
    save=True,
    save_period=1 ,
    resume=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Training device: cpu
WARNING ⚠️ model 'yolo12n.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/sidewalk_hazard_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, d

## 9. Validate best model

In [ ]:
best_model_path = "/content/drive/MyDrive/yolo12n_sidewalk_runs/yolo12n_sidewalk_hazard/weights/best.pt"

best_model = YOLO(best_model_path)
metrics = best_model.val(data=str(data_yaml_path), imgsz=640, device=training_device)